# ACPT / AllCPT (Danish)

Computes ACPT / AllCPT over BM25 and BGE-M3 dense (cw=8192, bs=32) for 27 selected queries on the `danish_final_cluster_r10/merged` augmented chunks corpus.

- **ACPT (Any Context Precedes Target):** `True` if any `utilized_context_chunk_ids` rank is strictly better (lower) than the target rank.
- **AllCPT (All Contexts Precede Target):** `True` if every utilized-context rank is strictly better than the target rank.

Ranks beyond top-100 (or missing) are treated as `inf`; `inf < inf` is `False`.

In [4]:
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from lcr.formatter import DataFormatter
from lcr.modeling.bm25_embedder import BM25Embedder
from lcr.modeling.bge_m3_embedder import BGEM3Embedder

DATA_DIR = Path("../data/processed/danish_final_cluster_r10/merged")
CHUNKS_PATH = DATA_DIR / "augmented_chunks.jsonl"
QUERIES_PATH = DATA_DIR / "contextual_queries.jsonl"

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BGE_CACHE = CACHE_DIR / "bge_m3_dense_augmented_chunks_danish.npz"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE)

TARGET_QIDS = [
    "aaregnskabsloven_§138a_stk3",
    "aaregnskabsloven_§137n_stk2",
    "aaregnskabsloven_§138a_stk4",
    "aaregnskabsloven_§97a_stk4",
    "almenboligloven_§58b_stk1",
    "almenboligloven_§98d_stk4",
    "almenboligloven_§169_stk2",
    "almenboligloven_§179a_stk4",
    "barnetslov_§49_stk3",
    "barnetslov_§201_stk1",
    "barnetslov_§139_stk5",
    "barnetslov_§13_stk5",
    "erhversfondsloven_§99_stk3",
    "erhversfondsloven_§119_stk3",
    "selskabsloven_§305_stk2",
    "selskabsloven_§37_stk2",
    "serviceloven_§82b_stk2",
    "serviceloven_§137e_stk2",
    "serviceloven_§97_stk8",
    "serviceloven_§73_stk2",
    "straffeloven_§132b_stk5",
    "straffeloven_§152c_stk1",
    "straffeloven_§236_stk10",
    "straffeloven_§183_stk2",
]
print("target queries:", len(TARGET_QIDS))

device: mps
target queries: 24


## Load + filter data

In [5]:
formatter = DataFormatter()
formatter.load_from_jsonl(CHUNKS_PATH, "documents")
formatter.load_from_jsonl(QUERIES_PATH, "queries")

target_set = set(TARGET_QIDS)
formatter.queries_dataset = formatter.queries_dataset.filter(lambda x: x["chunk_id"] in target_set)
print("filtered queries:", len(formatter.queries_dataset))
assert len(formatter.queries_dataset) == len(TARGET_QIDS), (
    f"expected {len(TARGET_QIDS)} queries, got {len(formatter.queries_dataset)}"
)

utilized_by_qid: dict[str, list[str]] = {
    row["chunk_id"]: list(row["utilized_context_chunk_ids"] or [])
    for row in formatter.queries_dataset
}
for qid in TARGET_QIDS:
    print(f"  {qid}: {len(utilized_by_qid[qid])} utilized context(s)")

Loaded documents dataset from jsonl ../data/processed/danish_final_cluster_r10/merged/augmented_chunks.jsonl
Loaded queries dataset from jsonl ../data/processed/danish_final_cluster_r10/merged/contextual_queries.jsonl
filtered queries: 24
  aaregnskabsloven_§138a_stk3: 2 utilized context(s)
  aaregnskabsloven_§137n_stk2: 2 utilized context(s)
  aaregnskabsloven_§138a_stk4: 2 utilized context(s)
  aaregnskabsloven_§97a_stk4: 1 utilized context(s)
  almenboligloven_§58b_stk1: 1 utilized context(s)
  almenboligloven_§98d_stk4: 2 utilized context(s)
  almenboligloven_§169_stk2: 2 utilized context(s)
  almenboligloven_§179a_stk4: 3 utilized context(s)
  barnetslov_§49_stk3: 1 utilized context(s)
  barnetslov_§201_stk1: 1 utilized context(s)
  barnetslov_§139_stk5: 2 utilized context(s)
  barnetslov_§13_stk5: 2 utilized context(s)
  erhversfondsloven_§99_stk3: 2 utilized context(s)
  erhversfondsloven_§119_stk3: 1 utilized context(s)
  selskabsloven_§305_stk2: 1 utilized context(s)
  selskab

## BM25 ranking

In [6]:
bm25 = BM25Embedder(spacy_model="da_core_news_sm")
t0 = time.time()
bm25_results, bm25_label_ids, _ = bm25.compute_results(formatter)
print(f"BM25 done in {time.time() - t0:.1f}s. queries: {len(bm25_label_ids)}, top-N per query: {len(next(iter(bm25_results.values())))}")

BM25 done in 75.3s. queries: 24, top-N per query: 100


## BGE-M3 dense ranking (cached doc embeddings)

In [7]:
bge = BGEM3Embedder(encoding_type="dense", batch_size=32, device=DEVICE)

chunks, chunk_ids = formatter.get_flattened()

if BGE_CACHE.exists():
    t0 = time.time()
    cache = np.load(BGE_CACHE, allow_pickle=False)
    doc_emb = cache["embeddings"]
    cached_ids = cache["chunk_ids"].tolist()
    assert cached_ids == list(chunk_ids), "cache chunk_ids don't match current corpus order; delete cache to rebuild"
    print(f"loaded BGE-M3 doc embeddings from cache in {time.time() - t0:.1f}s shape={doc_emb.shape}")
else:
    t0 = time.time()
    doc_emb = bge.embed_documents(chunks)
    doc_emb = np.asarray(doc_emb, dtype=np.float32)
    np.savez(BGE_CACHE, embeddings=doc_emb, chunk_ids=np.array(chunk_ids))
    print(f"encoded {len(chunks)} docs in {time.time() - t0:.1f}s, cached to {BGE_CACHE} shape={doc_emb.shape}")

pre tokenize: 100%|██████████| 242/242 [00:00<00:00, 766.61it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 242/242 [02:31<00:00,  1.60it/s]

encoded 7743 docs in 167.4s, cached to cache/bge_m3_dense_augmented_chunks_danish.npz shape=(7743, 1024)


In [8]:
queries, bge_label_ids = formatter.get_queries()
t0 = time.time()
q_emb = bge.embed_queries(queries)
q_emb = np.asarray(q_emb, dtype=np.float32)
print(f"encoded {len(queries)} queries in {time.time() - t0:.1f}s shape={q_emb.shape}")

q_t = bge._convert_to_tensor(q_emb)
d_t = bge._convert_to_tensor(doc_emb)
scores = bge.get_similarities(q_t, d_t).numpy()
bge_results = bge.get_results(scores, list(chunk_ids))
print(f"BGE-M3 ranking done. top-N per query: {len(next(iter(bge_results.values())))}")

encoded 24 queries in 0.9s shape=(24, 1024)
BGE-M3 ranking done. top-N per query: 100


## Compute ACPT / AllCPT

In [9]:
def ranks_from_results(per_query_scores: dict[str, float]) -> dict[str, int]:
    items = sorted(per_query_scores.items(), key=lambda kv: kv[1], reverse=True)
    return {cid: r for r, (cid, _) in enumerate(items, start=1)}


def rank_of(rank_map: dict[str, int], cid: str) -> float:
    return rank_map.get(cid, math.inf)


def fmt_rank(r: float) -> str:
    return "inf" if r == math.inf else str(int(r))


def metric_rows(retriever_name: str, results: dict, label_ids: list[str]) -> list[dict]:
    rows = []
    for i, qid in enumerate(label_ids):
        rank_map = ranks_from_results(results[str(i)])
        target_rank = rank_of(rank_map, qid)
        ctx_ids = utilized_by_qid.get(qid, [])
        ctx_ranks = [rank_of(rank_map, c) for c in ctx_ids]
        acpt = any(cr < target_rank for cr in ctx_ranks)
        allcpt = bool(ctx_ranks) and all(cr < target_rank for cr in ctx_ranks)
        rows.append({
            "retriever": retriever_name,
            "query_chunk_id": qid,
            "target_rank": fmt_rank(target_rank),
            "n_contexts": len(ctx_ids),
            "context_ranks": [fmt_rank(cr) for cr in ctx_ranks],
            "ACPT": acpt,
            "AllCPT": allcpt,
        })
    return rows


bm25_rows = metric_rows("bm25", bm25_results, bm25_label_ids)
bge_rows = metric_rows("bge-m3-dense", bge_results, bge_label_ids)
df_bm25 = pd.DataFrame(bm25_rows)
df_bge = pd.DataFrame(bge_rows)

### Per-query: BM25

In [10]:
df_bm25

,retriever,query_chunk_id,target_rank,n_contexts,context_ranks,ACPT,AllCPT
0,bm25,almenboligloven_§169_stk2,6,2,"[45, 5]",True,False
1,bm25,almenboligloven_§98d_stk4,inf,2,"[3, 5]",True,True
2,bm25,almenboligloven_§179a_stk4,82,3,"[1, inf, 2]",True,False
3,bm25,almenboligloven_§58b_stk1,19,1,[1],True,True
4,bm25,barnetslov_§49_stk3,3,1,[1],True,True
5,bm25,barnetslov_§13_stk5,3,2,"[8, 44]",False,False
6,bm25,barnetslov_§201_stk1,2,1,[1],True,True
7,bm25,barnetslov_§139_stk5,24,2,"[15, 3]",True,True
8,bm25,serviceloven_§97_stk8,9,1,[1],True,True
9,bm25,serviceloven_§137e_stk2,10,2,"[29, 4]",True,False


### Per-query: BGE-M3 dense

In [11]:
df_bge

,retriever,query_chunk_id,target_rank,n_contexts,context_ranks,ACPT,AllCPT
0,bge-m3-dense,almenboligloven_§169_stk2,1,2,"[inf, 12]",False,False
1,bge-m3-dense,almenboligloven_§98d_stk4,3,2,"[2, 1]",True,True
2,bge-m3-dense,almenboligloven_§179a_stk4,12,3,"[1, 23, 3]",True,False
3,bge-m3-dense,almenboligloven_§58b_stk1,35,1,[1],True,True
4,bge-m3-dense,barnetslov_§49_stk3,1,1,[2],False,False
5,bge-m3-dense,barnetslov_§13_stk5,7,2,"[9, inf]",False,False
6,bge-m3-dense,barnetslov_§201_stk1,2,1,[16],False,False
7,bge-m3-dense,barnetslov_§139_stk5,86,2,"[3, 5]",True,True
8,bge-m3-dense,serviceloven_§97_stk8,11,1,[1],True,True
9,bge-m3-dense,serviceloven_§137e_stk2,23,2,"[38, 2]",True,False


### Side-by-side

In [12]:
merged = (
    df_bm25.rename(columns={
        "target_rank": "target_rank_bm25",
        "context_ranks": "context_ranks_bm25",
        "ACPT": "ACPT_bm25",
        "AllCPT": "AllCPT_bm25",
    })[["query_chunk_id", "n_contexts", "target_rank_bm25", "context_ranks_bm25", "ACPT_bm25", "AllCPT_bm25"]]
    .merge(
        df_bge.rename(columns={
            "target_rank": "target_rank_bge",
            "context_ranks": "context_ranks_bge",
            "ACPT": "ACPT_bge",
            "AllCPT": "AllCPT_bge",
        })[["query_chunk_id", "target_rank_bge", "context_ranks_bge", "ACPT_bge", "AllCPT_bge"]],
        on="query_chunk_id",
    )
)
merged

,query_chunk_id,n_contexts,target_rank_bm25,context_ranks_bm25,ACPT_bm25,AllCPT_bm25,target_rank_bge,context_ranks_bge,ACPT_bge,AllCPT_bge
0,almenboligloven_§169_stk2,2,6,"[45, 5]",True,False,1,"[inf, 12]",False,False
1,almenboligloven_§98d_stk4,2,inf,"[3, 5]",True,True,3,"[2, 1]",True,True
2,almenboligloven_§179a_stk4,3,82,"[1, inf, 2]",True,False,12,"[1, 23, 3]",True,False
3,almenboligloven_§58b_stk1,1,19,[1],True,True,35,[1],True,True
4,barnetslov_§49_stk3,1,3,[1],True,True,1,[2],False,False
5,barnetslov_§13_stk5,2,3,"[8, 44]",False,False,7,"[9, inf]",False,False
6,barnetslov_§201_stk1,1,2,[1],True,True,2,[16],False,False
7,barnetslov_§139_stk5,2,24,"[15, 3]",True,True,86,"[3, 5]",True,True
8,serviceloven_§97_stk8,1,9,[1],True,True,11,[1],True,True
9,serviceloven_§137e_stk2,2,10,"[29, 4]",True,False,23,"[38, 2]",True,False


### Aggregate

In [13]:
agg = pd.DataFrame([
    {"retriever": "bm25", "ACPT_mean": df_bm25["ACPT"].mean(), "AllCPT_mean": df_bm25["AllCPT"].mean()},
    {"retriever": "bge-m3-dense", "ACPT_mean": df_bge["ACPT"].mean(), "AllCPT_mean": df_bge["AllCPT"].mean()},
])
agg

,retriever,ACPT_mean,AllCPT_mean
0,bm25,0.875000,0.666667
1,bge-m3-dense,0.666667,0.583333


In [15]:
# get counts instead of means for ACPT and AllCPT
agg_counts = pd.DataFrame([
    {"retriever": "bm25", "ACPT_count": df_bm25["ACPT"].sum(), "AllCPT_count": df_bm25["AllCPT"].sum()},
    {"retriever": "bge-m3-dense", "ACPT_count": df_bge["ACPT"].sum(), "AllCPT_count": df_bge["AllCPT"].sum()},
])
agg_counts

,retriever,ACPT_count,AllCPT_count
0,bm25,21,16
1,bge-m3-dense,16,14


### Solved counts (target in top-10)

In [14]:
def _num(r):
    return math.inf if r == "inf" else int(r)

bm25_solved = sum(_num(r) <= 10 for r in merged["target_rank_bm25"])
bge_solved = sum(_num(r) <= 10 for r in merged["target_rank_bge"])
either_solved = sum(
    (_num(a) <= 10) or (_num(b) <= 10)
    for a, b in zip(merged["target_rank_bm25"], merged["target_rank_bge"])
)
n_total = len(merged)
print(f"solved by BM25:           {bm25_solved}/{n_total}")
print(f"solved by BGE-M3 dense:   {bge_solved}/{n_total}")
print(f"solved by at least one:   {either_solved}/{n_total}")

solved by BM25:           13/24
solved by BGE-M3 dense:   13/24
solved by at least one:   16/24


In [16]:
# get ids of those that were were not solved by either
unsolved_ids = [
    qid for qid, a, b in zip(merged["query_chunk_id"], merged["target_rank_bm25"], merged["target_rank_bge"])
    if (_num(a) > 10) and (_num(b) > 10)
]
print("unsolved query chunk ids:")
for qid in unsolved_ids:
    print(f"  {qid}")

unsolved query chunk ids:
  almenboligloven_§179a_stk4
  almenboligloven_§58b_stk1
  barnetslov_§139_stk5
  serviceloven_§73_stk2
  serviceloven_§82b_stk2
  erhversfondsloven_§99_stk3
  erhversfondsloven_§119_stk3
  selskabsloven_§305_stk2
